In [0]:
# =============================================================================
# NOTEBOOK  : silver_pos_transactions_batch
# PURPOSE   : bronze.pos_transactions (batch rows) → silver.pos_transactions
# LAYER     : Silver (clean, type-cast, merge, delete cancelled transactions)
# FREQUENCY : Daily at 2 AM (JOB B)
# PATTERN   : spark.read + audit watermark (production pattern)
#             reads only new batch rows since last successful run
#             _source filter ensures stream rows never touched by this notebook
#
# MERGE & DEDUP LOGIC:
#   Source  : bronze.pos_transactions WHERE _source = 'pos_batch_csv'
#             AND ingested_at > last_successful_run_time
#   No within-batch dedup — batch CSV guaranteed clean by source system
#
#   MERGE cases:
#     Case 1: transaction_id IN silver, no data change  → IGNORE
#     Case 2: transaction_id NOT in silver              → INSERT
#     Case 3: transaction_id IN silver, data changed    → UPDATE
#             Changeable: total_amount, quantity, payment_method, channel
#             transaction_ts NOT updated — stream event time is authoritative
#             source_system NOT updated — preserves original stream source
#
#   DELETE (Case 4):
#     Scope: only transaction_dates actually present in this batch file
#     Handles: single-day, multi-day, and gap batch files correctly
#     Records in silver for those exact dates NOT in batch = cancelled
#     Uses anti-join — efficient and null-safe for large sets
#
# REPROCESSING:
#   Full reprocess  → delete last SUCCESS entry from pipeline_audit
#   Specific window → update end_time in pipeline_audit to desired start point
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, to_timestamp
)
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_pos_transactions"]
TARGET_TABLE = cfg["tables"]["silver_pos_transactions"]
PIPELINE     = "silver_pos_transactions_batch"

In [0]:
# ── Read + Transform + Merge + Delete ─────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    # ── INCREMENTAL READ ──────────────────────────────────────────────────────
    # Filter 1: _source = pos_batch_csv — never touch stream rows
    # Filter 2: ingested_at > last_run_time — only new batch rows
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("_source") == "pos_batch_csv")
            .filter(col("ingested_at") > lit(last_run_time))
        )
    else:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("_source") == "pos_batch_csv")
        )
 
    rows_read = bronze_df.count()
    print(f"[READ] {rows_read} new batch rows since last run")
 
    if rows_read == 0:
        print(f"[SKIP] No new batch rows to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0,
                  extra={"rows_inserted": "0", "rows_updated": "0",
                         "rows_deleted": "0"})
        raise SystemExit(0)
 
    # ── TRANSFORM ─────────────────────────────────────────────────────────────
    df = (
        bronze_df
 
        # Cast timestamp ISO string → TimestampType
        # Renamed transaction_ts — avoids Spark reserved word 'timestamp'
        .withColumn("transaction_ts",
            to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
 
        # Money columns Double → Decimal(10,2) for precision
        .withColumn("unit_price",
            col("unit_price").cast(DecimalType(10, 2)))
        .withColumn("total_amount",
            col("total_amount").cast(DecimalType(10, 2)))
 
        # transaction_date from event time — partition column
        # Use transaction_ts (when event happened) not ingested_at
        .withColumn("transaction_date",
            to_date(
                to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
            ).cast("string"))
 
        # Silver audit columns
        .withColumn("processed_at",    current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
        .withColumn("source_system",   lit("pos_batch_csv"))
 
        # Enforce silver schema — drops all bronze-only columns
        # (_source, source_file, ingested_date, pipeline_run_id from bronze, etc.)
        .select([f.name for f in SILVER_POS_TRANSACTIONS_SCHEMA.fields])
    )
 
    # ── MERGE INTO SILVER ──────────────────────────────────────────────────────
    # Case 3: matched + data changed → UPDATE changeable fields only
    # Case 2: not matched            → INSERT
    # Case 1: matched + no change    → no rule fires → IGNORE
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(
            df.alias("s"),
            "t.transaction_id = s.transaction_id"
        )
        .whenMatchedUpdate(
            condition="""
                t.total_amount   != s.total_amount   OR
                t.quantity       != s.quantity        OR
                t.payment_method != s.payment_method  OR
                t.channel        != s.channel
            """,
            set={
                "total_amount":    "s.total_amount",
                "quantity":        "s.quantity",
                "payment_method":  "s.payment_method",
                "channel":         "s.channel",
                "processed_at":    "current_timestamp()",
                "pipeline_run_id": f"'{run_id}'"
                # transaction_ts NOT updated — stream event time authoritative
                # source_system  NOT updated — preserve original stream source
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    merge_metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )
    rows_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(merge_metrics.get("numTargetRowsUpdated", 0))
 
    print(f"[MERGE] inserted={rows_inserted} | updated={rows_updated}")
 
    # ── DELETE CANCELLED TRANSACTIONS (Case 4) ────────────────────────────────
    # Scope: exact transaction_dates present in this batch file only
    # Handles: single-day, multi-day, and gap batch files (Day 1,2,4,5 not Day 3)
    # Day 3 records untouched — not in batch_dates → never in silver_in_scope
    # Anti-join used — efficient and null-safe vs NOT IN for large sets
    batch_dates   = df.select("transaction_date").distinct()
    batch_txn_ids = df.select("transaction_id").distinct()
 
    # Silver records whose transaction_date is covered by this batch
    silver_in_scope = (
        spark.read.table(TARGET_TABLE)
        .join(batch_dates, "transaction_date", "inner")
        .select("transaction_id", "transaction_date")
    )
 
    # Anti-join: in silver for those dates but NOT in batch = cancelled
    cancelled_txns = silver_in_scope.join(
        batch_txn_ids, "transaction_id", "left_anti"
    )
 
    rows_to_delete = cancelled_txns.count()
    rows_deleted   = 0
 
    if rows_to_delete > 0:
        cancelled_ids   = [r.transaction_id  for r in cancelled_txns.collect()]
        batch_date_list = [r.transaction_date for r in batch_dates.collect()]
 
        (
            DeltaTable.forName(spark, TARGET_TABLE)
            .delete(
                col("transaction_id").isin(cancelled_ids) &
                col("transaction_date").isin(batch_date_list)
            )
        )
 
        delete_metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_deleted = int(delete_metrics.get("numDeletedRows", 0))
        print(f"[DELETE] cancelled transactions removed: {rows_deleted}")
    else:
        print(f"[DELETE] no cancelled transactions found")
 
    rows_written = rows_inserted + rows_updated
    print(f"[DONE] rows_written={rows_written} | rows_deleted={rows_deleted}")
 
    end_audit(
        spark, PROJECT_ROOT, run_id, "SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
        rows_rejected=rows_deleted,
        extra={
            "rows_inserted": str(rows_inserted),
            "rows_updated":  str(rows_updated),
            "rows_deleted":  str(rows_deleted)
        }
    )
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── CELL 3: Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written",
            "rows_rejected", "extra_metadata") \
    .show(truncate=False)
 
# 2. Silver rows by source
print("Silver rows by source_system:")
(
    spark.read.table(TARGET_TABLE)
    .groupBy("source_system")
    .count()
    .show(truncate=False)
)
 
# 3. Nulls in key columns
print("Null transaction_ids:",
    spark.read.table(TARGET_TABLE)
    .filter(col("transaction_id").isNull())
    .count()
)
 
# 4. Delta history — last 3 to see both MERGE and DELETE operations
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 3") \
    .select("version", "operation", "operationMetrics") \
    .show(truncate=False)
 